## Check GPU

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Define Important PATHS

In [5]:
from pathlib import Path

# ============================================================
# CUSTOM ULTRALYTICS REPOSITORY
# ============================================================

CUSTOM_INPUT = Path(
    "/kaggle/input/datasets/justinalmadrones/ultralytics-kaggle3"
)

WORK_ROOT = Path(
    "/kaggle/working/ultralytics-custom"
)


# ============================================================
# DATASET
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/justinalmadrones/"
    "ultralytics-kaggle3/"
    "custom_models/training/finalnapls"
)


print("Custom repo exists:", CUSTOM_INPUT.exists())
print("Dataset exists:", DATASET_ROOT.exists())

Custom repo exists: True
Dataset exists: True


## Custom Ultralytics

In [6]:
import shutil

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

shutil.copytree(
    CUSTOM_INPUT,
    WORK_ROOT
)

print("Copied to:")
print(WORK_ROOT)

print("\nChecks:")
print(
    "pyproject.toml:",
    (WORK_ROOT / "pyproject.toml").exists()
)

print(
    "Ultralytics:",
    (WORK_ROOT / "ultralytics").is_dir()
)

print(
    "custom_models:",
    (WORK_ROOT / "custom_models").is_dir()
)

Copied to:
/kaggle/working/ultralytics-custom

Checks:
pyproject.toml: True
Ultralytics: True
custom_models: True


## Check for YAML file

In [7]:
from pathlib import Path

MODEL_YAML = (
    WORK_ROOT
    / "custom_models"
    / "yolov8s-ca-gelu-direct-p2-seg.yaml"
)

print("Model YAML:")
print(MODEL_YAML)

print("\nExists:")
print(MODEL_YAML.exists())

assert MODEL_YAML.exists(), (
    f"Model YAML not found:\n{MODEL_YAML}"
)

print("\n✅ Direct-P2 YAML found.")

Model YAML:
/kaggle/working/ultralytics-custom/custom_models/yolov8s-ca-gelu-direct-p2-seg.yaml

Exists:
True

✅ Direct-P2 YAML found.


## Custom Ultralytics 

In [8]:
%cd /kaggle/working/ultralytics-custom

/kaggle/working/ultralytics-custom


In [9]:
!pip uninstall -y ultralytics
!pip install -e . --no-deps

Obtaining file:///kaggle/working/ultralytics-custom
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.117-0.editable-py3-none-any.whl size=4924 sha256=921eff7510b5f515f7dda5de91687b657a5c8d4a07c36900a303afbc752c95be
  Stored in directory: /tmp/pip-ephem-wheel-cache-zynj44lb/wheels/4d/b1/0b/b09255d3018d9b20cdfc42feff8e6167781a4b6cbac58e6c4c
Successfully built ultralytics


## Verify Correct Ultralytics

In [10]:
import sys

CUSTOM_REPO = (
    "/kaggle/working/ultralytics-custom"
)

if CUSTOM_REPO not in sys.path:
    sys.path.insert(0, CUSTOM_REPO)


import ultralytics

print(
    "Version:",
    ultralytics.__version__
)

print(
    "Loaded from:",
    ultralytics.__file__
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Version: 8.4.117
Loaded from: /kaggle/working/ultralytics-custom/ultralytics/__init__.py


## Verify Coordinate Attention

In [11]:
from ultralytics.nn.modules import CoordinateAttention

print(
    "CoordinateAttention:",
    CoordinateAttention
)

CoordinateAttention: <class 'ultralytics.nn.modules.block.CoordinateAttention'>


## Build new Model

In [12]:
from ultralytics import YOLO

print("Model YAML:")
print(MODEL_YAML)

print(
    "\nExists:",
    MODEL_YAML.exists()
)


model = YOLO(
    str(MODEL_YAML),
    task="segment"
)


model.info(
    verbose=True
)

Model YAML:
/kaggle/working/ultralytics-custom/custom_models/yolov8s-ca-gelu-direct-p2-seg.yaml

Exists: True
YOLOv8s-ca-gelu-direct-p2-seg summary: 203 layers, 13,299,924 parameters, 13,299,908 gradients


(203, 13299924, 13299908, 0.0)

## Verify CA + GELU

In [13]:
import torch.nn as nn

from ultralytics.nn.modules import (
    Conv,
    CoordinateAttention
)


total_conv = 0
gelu_conv = 0
silu_conv = 0
identity_conv = 0
other_conv = 0

ca_count = 0
hardswish_count = 0


for name, module in model.model.named_modules():

    if isinstance(module, Conv):

        total_conv += 1

        if isinstance(
            module.act,
            nn.GELU
        ):
            gelu_conv += 1

        elif isinstance(
            module.act,
            nn.SiLU
        ):
            silu_conv += 1

        elif isinstance(
            module.act,
            nn.Identity
        ):
            identity_conv += 1

        else:
            other_conv += 1


    if isinstance(
        module,
        CoordinateAttention
    ):
        ca_count += 1
        print(
            "Coordinate Attention:",
            name
        )


    if isinstance(
        module,
        nn.Hardswish
    ):
        hardswish_count += 1


print("\n" + "=" * 60)
print("ARCHITECTURE CHECK")
print("=" * 60)

print(
    "Total Conv:",
    total_conv
)

print(
    "GELU:",
    gelu_conv
)

print(
    "SiLU:",
    silu_conv
)

print(
    "Identity:",
    identity_conv
)

print(
    "Other:",
    other_conv
)

print(
    "Coordinate Attention:",
    ca_count
)

print(
    "Hardswish:",
    hardswish_count
)


assert silu_conv == 0

assert other_conv == 0

assert ca_count == 3

assert (
    gelu_conv
    + identity_conv
    == total_conv
)


print(
    "\n✅ CA + GLOBAL GELU VERIFIED"
)

Coordinate Attention: model.5
Coordinate Attention: model.8
Coordinate Attention: model.12

ARCHITECTURE CHECK
Total Conv: 81
GELU: 80
SiLU: 0
Identity: 1
Other: 0
Coordinate Attention: 3
Hardswish: 3

✅ CA + GLOBAL GELU VERIFIED


## Verify P2 Head

In [14]:
head = model.model.model[-1]

print("=" * 60)
print("SEGMENT HEAD")
print("=" * 60)

print(
    "Head:",
    type(head).__name__
)

print(
    "From layers:",
    head.f
)

print(
    "Strides:",
    head.stride
)


assert list(head.f) == [
    24,
    21,
    27,
    30
]


print(
    "\n✅ Direct-P2 Segment head verified."
)

SEGMENT HEAD
Head: Segment
From layers: [24, 21, 27, 30]
Strides: tensor([ 8.,  4., 16., 32.])

✅ Direct-P2 Segment head verified.


## GPU Forward Test

In [15]:
device = torch.device(
    "cuda:0"
)

model.model.to(device)
model.model.eval()


dummy = torch.randn(
    1,
    3,
    640,
    640,
    device=device
)


with torch.no_grad():

    _ = model.model(dummy)


print(
    "✅ GPU forward pass successful"
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

✅ GPU forward pass successful
GPU: Tesla T4


## Dataset Check

In [16]:
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/justinalmadrones/"
    "ultralytics-kaggle3/"
    "custom_models/training/finalnapls"
)

print("Dataset root:")
print(DATASET_ROOT)

print("\nExists:", DATASET_ROOT.exists())

print("Train images:",
      (DATASET_ROOT / "train/images").exists())

print("Train labels:",
      (DATASET_ROOT / "train/labels").exists())

print("Val images:",
      (DATASET_ROOT / "val/images").exists())

print("Val labels:",
      (DATASET_ROOT / "val/labels").exists())

Dataset root:
/kaggle/input/datasets/justinalmadrones/ultralytics-kaggle3/custom_models/training/finalnapls

Exists: True
Train images: True
Train labels: True
Val images: True
Val labels: True


## Data YAML

In [17]:
import yaml
from pathlib import Path

DATA_YAML = Path(
    "/kaggle/working/pothole_data.yaml"
)

data_config = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "names": {
        0: "pothole"
    }
}

# Add test split only if available
if (DATASET_ROOT / "test/images").exists():
    data_config["test"] = "test/images"

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False
    )

print("=" * 60)
print("NEW KAGGLE DATA.YAML")
print("=" * 60)

print(DATA_YAML.read_text())

print("Exists:", DATA_YAML.exists())

NEW KAGGLE DATA.YAML
path: /kaggle/input/datasets/justinalmadrones/ultralytics-kaggle3/custom_models/training/finalnapls
train: train/images
val: val/images
names:
  0: pothole

Exists: True


In [18]:
import yaml

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

root = Path(cfg["path"])

train_path = root / cfg["train"]
val_path = root / cfg["val"]

print("Train:")
print(train_path)
print("Exists:", train_path.exists())

print("\nVal:")
print(val_path)
print("Exists:", val_path.exists())

assert train_path.exists(), "Training images not found!"
assert val_path.exists(), "Validation images not found!"

print("\n✅ Dataset YAML verified.")

Train:
/kaggle/input/datasets/justinalmadrones/ultralytics-kaggle3/custom_models/training/finalnapls/train/images
Exists: True

Val:
/kaggle/input/datasets/justinalmadrones/ultralytics-kaggle3/custom_models/training/finalnapls/val/images
Exists: True

✅ Dataset YAML verified.


## Validate Segmentation Labels

In [19]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
}


def validate_split(
    root,
    split
):

    images_dir = (
        root
        / split
        / "images"
    )

    labels_dir = (
        root
        / split
        / "labels"
    )


    images = [
        p
        for p in images_dir.iterdir()
        if p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]


    labels = list(
        labels_dir.glob("*.txt")
    )


    invalid = []

    empty_labels = 0


    for label_file in labels:

        text = (
            label_file
            .read_text(
                encoding="utf-8"
            )
            .strip()
        )


        if not text:

            empty_labels += 1

            continue


        for line_number, line in enumerate(
            text.splitlines(),
            start=1
        ):

            parts = (
                line
                .strip()
                .split()
            )


            if len(parts) < 7:

                invalid.append(
                    (
                        label_file.name,
                        line_number,
                        "too few values"
                    )
                )

                continue


            try:

                class_id = int(
                    float(
                        parts[0]
                    )
                )


                coords = list(
                    map(
                        float,
                        parts[1:]
                    )
                )


            except ValueError:

                invalid.append(
                    (
                        label_file.name,
                        line_number,
                        "non-numeric"
                    )
                )

                continue


            if class_id != 0:

                invalid.append(
                    (
                        label_file.name,
                        line_number,
                        f"class {class_id}"
                    )
                )


            if len(coords) % 2 != 0:

                invalid.append(
                    (
                        label_file.name,
                        line_number,
                        "odd coordinate count"
                    )
                )


            if any(
                value < 0.0
                or value > 1.0
                for value
                in coords
            ):

                invalid.append(
                    (
                        label_file.name,
                        line_number,
                        "coordinate outside 0-1"
                    )
                )


    print("\n" + "=" * 60)

    print(
        split.upper()
    )

    print("=" * 60)

    print(
        "Images:",
        len(images)
    )

    print(
        "Labels:",
        len(labels)
    )

    print(
        "Backgrounds:",
        empty_labels
    )

    print(
        "Invalid:",
        len(invalid)
    )


    return invalid


train_invalid = validate_split(
    DATASET_ROOT,
    "train"
)


val_invalid = validate_split(
    DATASET_ROOT,
    "val"
)


if (
    train_invalid
    or val_invalid
):

    raise ValueError(
        "Invalid labels detected."
    )


print(
    "\n✅ DATASET LABEL CHECK PASSED"
)


TRAIN
Images: 2390
Labels: 2390
Backgrounds: 354
Invalid: 0

VAL
Images: 581
Labels: 581
Backgrounds: 80
Invalid: 0

✅ DATASET LABEL CHECK PASSED


## Training

In [20]:
from ultralytics import YOLO

model = YOLO(
    str(MODEL_YAML),
    task="segment"
)


results = model.train(

    data=str(DATA_YAML),

    # ==============================
    # TRAINING LENGTH
    # ==============================

    epochs=300,
    patience=30,

    # ==============================
    # IMAGE
    # ==============================

    imgsz=640,
    batch=4,

    # ==============================
    # HARDWARE
    # ==============================

    device=0,
    workers=2,

    # ==============================
    # OPTIMIZER
    # ==============================

    optimizer="AdamW",

    lr0=0.001,
    lrf=0.01,
    cos_lr=True,

    momentum=0.937,

    weight_decay=0.0005,

    warmup_epochs=3,

    # ==============================
    # REPRODUCIBILITY
    # ==============================

    seed=42,
    deterministic=True,

    # ==============================
    # FROM SCRATCH
    # ==============================

    pretrained=False,
    amp=True,

    # ==============================
    # SEGMENTATION
    # ==============================

    mask_ratio=4,
    overlap_mask=True,

    # ==============================

    # ==============================
    # VALIDATION
    # ==============================

    val=True,
    plots=True,

    # ==============================
    # SAVING
    # ==============================

    save=True,
    save_period=10,

    # ==============================
    # OUTPUT
    # ==============================

    project=(
        "/kaggle/working/"
        "runs/segment"
    ),

    name=(
        "yolov8s-ca-gelu-"
        "direct-p2-300"
    ),

    exist_ok=False
)

New https://pypi.org/project/ultralytics/8.4.123 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/pothole_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

## Best Segmentation Epoch

In [26]:
import pandas as pd

RUN_DIR = Path(
    "/kaggle/working/runs/"
    "segment/"
    "yolov8s-ca-gelu-"
    "direct-p2-300"
)


CSV_PATH = (
    RUN_DIR
    / "results.csv"
)


df = pd.read_csv(
    CSV_PATH
)


df.columns = (
    df.columns
    .str.strip()
)


best_idx = (
    df[
        "metrics/mAP50-95(M)"
    ]
    .idxmax()
)


best = df.loc[
    best_idx
]


print("=" * 60)

print(
    "BEST SEGMENTATION RESULT"
)

print("=" * 60)


print(
    "Epoch:",
    int(
        best["epoch"]
    )
)


print(
    "Mask Precision:",
    f'{best["metrics/precision(M)"] * 100:.2f}%'
)


print(
    "Mask Recall:",
    f'{best["metrics/recall(M)"] * 100:.2f}%'
)


print(
    "Mask mAP50:",
    f'{best["metrics/mAP50(M)"] * 100:.2f}%'
)


print(
    "Mask mAP50-95:",
    f'{best["metrics/mAP50-95(M)"] * 100:.2f}%'
)


print("\nBOX")


print(
    "Box Precision:",
    f'{best["metrics/precision(B)"] * 100:.2f}%'
)


print(
    "Box Recall:",
    f'{best["metrics/recall(B)"] * 100:.2f}%'
)


print(
    "Box mAP50:",
    f'{best["metrics/mAP50(B)"] * 100:.2f}%'
)


print(
    "Box mAP50-95:",
    f'{best["metrics/mAP50-95(B)"] * 100:.2f}%'
)

BEST SEGMENTATION RESULT
Epoch: 228
Mask Precision: 86.59%
Mask Recall: 78.18%
Mask mAP50: 83.98%
Mask mAP50-95: 55.94%

BOX
Box Precision: 85.79%
Box Recall: 77.46%
Box mAP50: 83.17%
Box mAP50-95: 58.05%


## ZIP

In [ ]:
import shutil

RESULT_ZIP = Path(
    "/kaggle/working/"
    "yolov8s-ca-gelu-"
    "direct-p2-300-results"
)


shutil.make_archive(
    str(RESULT_ZIP),
    "zip",
    str(RUN_DIR)
)


print(
    "✅ Results ZIP:"
)

print(
    str(RESULT_ZIP)
    + ".zip"
)

In [23]:
SOURCE_ZIP = Path(
    "/kaggle/working/"
    "ultralytics-ca-gelu-"
    "direct-p2-source"
)


shutil.make_archive(
    str(SOURCE_ZIP),
    "zip",
    str(WORK_ROOT)
)


print(
    "✅ Source ZIP:"
)

print(
    str(SOURCE_ZIP)
    + ".zip"
)

✅ Source ZIP:
/kaggle/working/ultralytics-ca-gelu-direct-p2-source.zip


In [33]:
%cd /kaggle/working

/kaggle/working


In [36]:
from IPython.display import FileLink, display

display(
    FileLink(
        "ultralytics-ca-gelu-direct-p2-source.zip"
    )
)

/kaggle/working/ultralytics-ca-gelu-direct-p2-source.zip

In [31]:
from IPython.display import FileLink
FileLink(r'yolov8s-ca-gelu-direct-p2-source.zip')

/kaggle/working/yolov8s-ca-gelu-direct-p2-source.zip